# Phương pháp lặp tựa Newton
# Thuật toán

**Input**: $A_{n \times n}$, $\varepsilon$

**Output**: $A^{-1}$

#### Thuật toán
1. Kiểm tra nếu $\det(A) = 0 \Rightarrow$ không tồn tại $A^{-1}$, dừng chương trình.

2. Khởi tạo $X_0 = \dfrac{A^T}{||A||_1 \cdot ||A||_\infty}$, $k=1$

3. Lặp:

    - Tính $X_{k} = X_{k-1}(2I - A X_{k-1})$

    - Tính $\Delta_k = ||I - A X_{k}||_\infty$

    Lặp cho đến khi $\Delta_k < \varepsilon$

4. In ra $X^* = A^{-1}$


In [5]:
import numpy as np
from sympy import Matrix
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
import asyncio
import sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

def output_matrix(X, precision=4):
    X = np.array(X)
    if np.iscomplexobj(X):
        display(Matrix(np.round(X.real, precision) + 1j * np.round(X.imag, precision)))
    else:
        display(Matrix(np.round(X, precision)))

def newton_schulz_inverse(A: np.ndarray, eps=1e-6, eta=None, max_iter: int = 100) -> 'np.ndarray | None':
    n = A.shape[0]
    if A.shape[0] != A.shape[1]:
        print("Lỗi: Ma trận không vuông.")
        return None
        
    det_A = np.linalg.det(A)
    if abs(det_A) < 1e-12:
        print("Lỗi: Ma trận suy biến (det = 0), không tồn tại ma trận nghịch đảo.")
        return None

    if (eps is None) == (eta is None):
        raise ValueError("Chỉ được chỉ định đúng 1 loại sai số: tuyệt đối (eps) hoặc tương đối (eta).")

    # Bước 2: Khởi tạo X0
    norm_1 = np.linalg.norm(A, ord=1)       
    norm_inf = np.linalg.norm(A, ord=np.inf) 
    alpha = 1.0 / (norm_1 * norm_inf)
    
    X_k = alpha * A.T
    I = np.eye(n)
    
    print("X_0 =")
    output_matrix(X_k)
    
    # Bước 3: Lặp
    for k in range(1, max_iter + 1):
        # Tính X_k mới
        X_next = X_k @ (2 * I - A @ X_k)
        
        # Tính sai số Delta_k
        if eps is not None:
            error = np.linalg.norm(I - A @ X_next, ord=np.inf)
            tol = eps
            err_type = "tuyệt đối"
        else:
            error = np.linalg.norm(X_next - X_k, ord=np.inf) / np.linalg.norm(X_next, ord=np.inf)
            tol = eta
            err_type = "tương đối"
        
        print(f"\n--- Vòng lặp k = {k} ---")
        print(f"X_{k} =")
        output_matrix(X_next)
        
        err_str = f"{error:.4e}" if error < 1e-4 else f"{error:.6f}"
        print(f"Sai số Delta_{k} = {err_str}")
        
        X_k = X_next
        
        if tol is not None and error <= tol:
            print(f"\n==> Hội tụ sau {k} bước lặp (sai số {err_type} <= {tol}).")
            break
    else:
        print(f"\n==> Đạt giới hạn lặp tối đa {max_iter} bước.")

    return X_k


In [6]:
def read_matrix_from_file(filename: str) -> 'np.ndarray | None':
    try:
        matrix = np.loadtxt(filename)
        if matrix.ndim == 1:
            matrix = matrix.reshape(1, -1)
        return matrix
    except FileNotFoundError:
        print(f"Không tìm thấy file {filename}")
        return None
    except Exception as e:
        print(f"Lỗi khi đọc file: {str(e)}")
        return None

# Đọc ma trận từ file
filename = "A.txt" 
A_file = read_matrix_from_file(filename)

if A_file is not None:
    print("Ma trận gốc A:")
    output_matrix(A_file)
    
    A_inv = newton_schulz_inverse(A_file, eps=1e-8)
    
    if A_inv is not None:
        print("\nKết quả A^{-1} (Newton-Schulz):")
        output_matrix(A_inv)
        
        print("\nKiểm chứng lại bằng np.linalg.inv(A):")
        output_matrix(np.linalg.inv(A_file))


Ma trận gốc A:


Matrix([
[4.0,  1.0,  1.0],
[1.0,  3.0, -1.0],
[2.0, -1.0,  5.0]])

X_0 =


Matrix([
[0.0714,  0.0179,  0.0357],
[0.0179,  0.0536, -0.0179],
[0.0179, -0.0179,  0.0893]])


--- Vòng lặp k = 1 ---
X_1 =


...

Matrix([
[ 0.3333, -0.1429, -0.0952],
[-0.1667,  0.4286,   0.119],
[-0.1667,  0.1429,  0.2619]])

Sai số Delta_9 = 1.0271e-10

==> Hội tụ sau 9 bước lặp (sai số tuyệt đối <= 1e-08).

Kết quả A^{-1} (Newton-Schulz):


Matrix([
[ 0.3333, -0.1429, -0.0952],
[-0.1667,  0.4286,   0.119],
[-0.1667,  0.1429,  0.2619]])


Kiểm chứng lại bằng np.linalg.inv(A):


Matrix([
[ 0.3333, -0.1429, -0.0952],
[-0.1667,  0.4286,   0.119],
[-0.1667,  0.1429,  0.2619]])